# GaugePredict CNN–LSTM Training Notebook  
Caitlin R. R. Turner, December 2025  

This notebook trains the GaugePredict CNN–LSTM forecasting model for water level at the Army Corps of Engineers water level gauge  
**Bonnet Carré Spillway @ Mississippi River (01280)**.

It provides a reproducible workflow to:

- Specify a target gauge and variable to be predicted (water level).
- Load basin-scale predictor datasets from a cached GaugePredict site dictionary.
- Select predictor sites either from all available gauges or from SHAP-selected subsets.
- Train and evaluate the CNN–LSTM model at one or more forecast horizons.
- Save per-horizon outputs and a compute summary (metrics, hyperparameters, runtimes, and hardware info).

The outputs are structured to plug directly into downstream GaugePredict routines for evaluation, plotting, and SHAP analysis.


## Import Necessary Dependencies

In [ ]:
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import torch

from GaugePredict.predict import (
    get_allowed_sites_for_horizon,
    horizon_dir,
    run_horizon,
    save_compute_summary,
    update_compute_summary,
    get_hardware_info,
)

from GaugePredict.routines import get_project_root, resolve_under_project


## Run Configuration
Here we define each run, how predictor gauges are selected, what is being predicted, and where inputs and outputs live on disk. These settings control reproducibility, file naming, and the exact datasets used in training.

### Run name

- **`run_name`**: A short label used to name the output folder for this experiment. Use this to keep runs separate when you are testing hyperparameters or different predictor sets.

In [ ]:
run_name = "function_test"

### Site selection and SHAP controls

These options control which upstream predictor gauges are used for training and whether SHAP is computed during the run.

- **`site_selection_mode`**: Determines how the set of predictor sites is chosen.
  - `"all"`: use every site available in the predictor JSON.
  - `"from_shap"`: use a SHAP-ranked subset of sites, loaded from `full_shap_root`.

- **`shap_mode`**: Determines whether SHAP is computed during this run.
  - `"none"`: do not compute SHAP in this run.
  - `"run"`: compute SHAP (typically slower, used when generating or updating SHAP rankings).

In [ ]:
site_selection_mode = "from_shap"  
shap_mode = "none"  

### Target definition

These fields define the prediction target and its metadata.

- **`target_site`**: Gauge identifier for the site being predicted.
- **`target_variable`**: Label for the target variable (used for organization and metadata).
- **`target_parameter_code`**: Parameter code for the target variable.
- **`target_units`**: Unit system for the target (typically `"metric"` or `"customary"`).

If using a csv file (as in this case):
- **`use_csv_target`**: If `True`, use `csv_path` as the target time series source.
- **`csv_path`**: Path to the CSV file containing the target data.
- **`csv_date_col`**: Name of the column containing dates.
- **`csv_value_col`**: Name of the column containing the target values (water level here).


In [ ]:
target_site = "01280" # We dont end up using this since we upload a csv file, but this is where a USGS site code would go
target_variable = "water_level"
target_parameter_code = "30210" # We dont end up using this since we upload a csv file, but this is where a USGS parameter code would go
target_units = "metric"

# If use_csv_target = True, define the following
use_csv_target = True
csv_path = examples_dir / "bcs_wl.csv"
csv_date_col = "date"
csv_value_col = "wl"

### Predictor dataset definition
These fields define where the predictor dataset is using the downloader notebook.
- **`predictor_type`**: Which predictor variable family to use (for example `"discharge"`).


In [ ]:
predictor_type = "discharge"

### Project paths
These fields define where to find input datasets and where to write outputs. All paths are resolved under the project root later in the notebook.
- **`examples_dir`**: Top-level examples directory for this project (inputs and outputs are typically stored here).
- **`results_dir`**: Where model outputs are written (per-run and per-horizon folders live under this).
-  **`predictor_json_root`**: Folder containing the cached predictor site dictionary for that predictor type. The code expects a JSON like `site_dict_<predictor_type>.json` inside this folder.
- **`full_shap_root`**: Folder containing SHAP artifacts for the target site (and potentially multiple horizons), used to determine the SHAP-ranked subset.


In [ ]:
examples_dir = Path("examples")
predictor_json_root = examples_dir / f"cached_data_{predictor_type}"
full_shap_root = results_dir / f"{target_site}_function_test_full" # contains list of all ranked shap sites
results_dir = examples_dir / "results"

### Time window and timezone

These fields define the modeling period. All predictors and the target are aligned to this window.

- **`start_date`**: Start of the training window (inclusive).
- **`end_date`**: End of the training window (inclusive).
- **`tz`**: Timezone used when interpreting dates and aligning time indices.


In [ ]:
start_date = "2005-01-01"
end_date = "2024-12-31"
tz = "UTC"

## Forecast horizons, SHAP subset sizes, and hyperparameters

Now we are going to define which forecast lead times are trained, how many predictor gauges are used when selecting sites from SHAP rankings, and the default and horizon-specific hyperparameters used by the CNN–LSTM model.

### Forecast horizons to run

- **`horizons`**: List of forecast horizons (in days) to train and evaluate.  
  Only the horizons listed here are executed in the training loop.

In this notebook we will run **1-day** and **3-day** forecasts:
- `horizons = [1, 3]`

Other horizons (5, 10, 15, 20, 30) are kept in the configuration below so the notebook remains a single reference template. To run them later, add them to the `horizons` list.

In [ ]:
horizons = [1, 3]
default_n_shap = 1950

### SHAP-based site subset sizes

When `site_selection_mode = "from_shap"`, GaugePredict can restrict the predictor gauges to a SHAP-ranked subset. This is controlled by `n_shap_by_h`.

- **`default_n_shap`**: Fallback number of SHAP-ranked sites to use if a horizon is not present in `n_shap_by_h`.
- **`n_shap_by_h`**: Dictionary mapping forecast horizon -> number of SHAP-selected predictor sites used for that horizon.

For this run, the best configuration uses:
- Horizon 1 uses 5 sites
- Horizon 3 uses 9 sites

In [ ]:
n_shap_by_h = {
    1: 34,
    3: 32,
    5: 45,
    10: 50,
    15: 65,
    20: 75,
    30: 100,
}

### Default hyperparameters

- **`hp_defaults`**: Baseline hyperparameters applied to all horizons first.  
  For each horizon, these defaults are copied and then overwritten by any horizon-specific values in `hp_by_h`.

Defaults:
- **`sequence_length`**: Number of days in each input sequence (lookback window).
- **`learning_rate`**: Optimizer learning rate.
- **`weight_decay`**: L2 regularization strength.
- **`max_grad_norm`**: Gradient clipping threshold to stabilize training.
- **`dropout_cnn`**, **`dropout_lstm`**, **`dropout_fc`**: Dropout rates in the CNN block, LSTM block, and fully connected head.
- **`epochs`**: Number of training epochs. This is set to 25 here.  
  For longer horizons (greater than H=5 days), training often benefits from running longer (for example 100 epochs).
- **`batch_size`**: Batch size for training.
- **`background_size`** and **`nsamples`**: SHAP-related settings when SHAP is enabled.
- **`cutoff_date`**: Date used to split training and testing (depends on your internal GaugePredict split logic).
- **`loss_function`**: Training loss (MSE here).

### Horizon-specific hyperparameters

- **`hp_by_h`**: Dictionary mapping forecast horizon -> hyperparameter overrides.  
  If a horizon appears in `hp_by_h`, those values replace the defaults for that horizon only.

In [ ]:
hp_defaults = dict(
    sequence_length=60,
    learning_rate=1.0e-4,
    weight_decay=1.0e-3,
    max_grad_norm=1.0,
    dropout_cnn=0.00,
    dropout_lstm=0.20,
    dropout_fc=0.0,
    epochs=5,
    batch_size=128,
    background_size=32,
    nsamples=128,
    cutoff_date=np.datetime64("2020-01-01"),
    loss_function=torch.nn.MSELoss(),
)

hp_by_h = {
    1: dict(sequence_length=3, learning_rate=1.5e-5, dropout_cnn=0.02, dropout_lstm=0.0),
    3: dict(sequence_length=4, learning_rate=9.5e-6, dropout_cnn=0.02, dropout_lstm=0.0),
    5: dict(sequence_length=4, learning_rate=7.0e-6, dropout_cnn=0.04, dropout_lstm=0.06),
    10: dict(sequence_length=5, learning_rate=5.0e-6, dropout_cnn=0.5, dropout_lstm=0.08),
    15: dict(sequence_length=5, learning_rate=2.5e-6, dropout_cnn=0.5, dropout_lstm=0.00),
    20: dict(sequence_length=7, learning_rate=0.95e-6, dropout_cnn=0.1, dropout_lstm=0.15, dropout_fc=0.1),
    30: dict(sequence_length=7, learning_rate=0.675e-6, dropout_cnn=0.1, dropout_lstm=0.125, dropout_fc=0.01),
}

## Resolve project paths and validate inputs
Here we will build file paths for all inputs and outputs used in the run. The goal is to make the notebook portable while keeping a consistent project structure to make it easier to use.

### Expected folder layout

This notebook assumes a GaugePredict-style layout where:

- `examples/` contains cached predictor datasets (for example, `examples/cached_data_discharge/`).
- `examples/results/` stores model outputs.
- A predictor site dictionary exists at:
  - `examples/cached_data_<predictor_type>/site_dict_<predictor_type>.json`
- If `use_csv_target = True`, a target CSV exists under `examples/` (or wherever `csv_path` points).

### Notebook path handling

In a Python script, `get_project_root(__file__, ...)` can use the script location to infer the project root. In a notebook, `__file__` is not defined, so we use the current working directory instead:

- **`Path.cwd()`**: the directory the notebook is running from.
- **`get_project_root(..., levels_up=1)`**: walks upward from `Path.cwd()` to locate the project root.  
  If your notebook is not inside the repository, set `project_root` manually.

Example manual override:
 **`project_root = Path(r"C:/path/to/GaugePredict").resolve()`**

In [ ]:
project_root = get_project_root(Path.cwd(), levels_up=1)

### Outputs directory

- **`results_root`**: Run-specific output directory. This is where all horizon folders and summary files are written.  
  It is created if it does not already exist.

In [ ]:
results_root = resolve_under_project(project_root, results_dir / f"{target_site}_{run_name}")
results_root.mkdir(parents=True, exist_ok=True)

### Target dataset paths

- **`csv_path`**: Absolute path to the target CSV (only if `use_csv_target = True`).

In [ ]:

if use_csv_target and (csv_path is None or not csv_path.exists()):
    raise FileNotFoundError(f"Target CSV not found: {csv_path}")
if use_csv_target:
    csv_path = resolve_under_project(project_root, csv_path)
else:
    csv_path = None


### Predictor JSON and validation
- **`full_shap_root`**: Absolute path to the SHAP results folder (only required when using SHAP-based site selection for predictor downsampling).
- **`predictor_json_root`**: Absolute path to the cached predictor dataset folder.
- **`json_path`**: Full path to the predictor site dictionary JSON, expected to be named:  
  - `site_dict_<predictor_type>.json`

If json is missing, the notebook raises `FileNotFoundError` immediately so the run does not start with misconfigured inputs.

In [ ]:
full_shap_root = resolve_under_project(project_root, full_shap_root)
predictor_json_root = resolve_under_project(project_root, predictor_json_root)
json_path = predictor_json_root / f"site_dict_{predictor_type}.json"
if not json_path.exists():
    raise FileNotFoundError(f"Predictor JSON not found: {json_path}")

### Data files list
- **`data_files`**: List of input datasets passed into `run_horizon(...)`.  
  Each entry includes a path and a key indicating what field to read from the file (here, `"parameter"`).

We will then double check all of the paths.

In [ ]:
data_files = [{"path": json_path, "data_key": "parameter"}]
print("project_root:", project_root)
print("results_root:", results_root)
print("predictor_json_path:", json_path)
if use_csv_target:
    print("target_csv_path:", csv_path)

## Record hardware and run metadata

Now we will record the run context needed to reproduce results and interpret runtimes, which is often requested in reporting. It chooses the compute device (GPU via CUDA when available, otherwise CPU), gathers basic hardware information, and assembles a `compute_summary` dictionary that is updated after each horizon finishes. 

### Device selection

- **`device`**: Selects `"cuda"` if a CUDA-enabled GPU is available to PyTorch, otherwise defaults to `"cpu"`.  
  This controls where the model and tensors are placed during training.

### Hardware information

- **`hardware_info`**: System and device metadata returned by `get_hardware_info(device)` (for example, GPU name, CPU info, memory, and related details depending on what the function reports).  
  This is saved with the results so training time differences can be compared across machines. This is currently uncommented for privacy, but you may turn it on at your own discrecion. 

### Compute summary dictionary

- **`compute_summary`**: A structured record of:
  - When the run was executed (**`run_timestamp`**, in UTC).
  - What was trained (target site, variable, parameter code, unit system).
  - Which predictors and time window were used (predictor type, dates, timezone).
  - Which horizons were run and how sites were selected (SHAP or all sites).
  - The resolved input and output paths used by the run.
  - The hardware metadata and a per-horizon results container.

Important fields:

- **`paths`**: Stores resolved absolute paths used during the run:
  - `examples_dir`, `project_root`, `results_root`
  - `predictor_json_root`, `predictor_json_path`
  - `full_shap_root`
  - Target CSV settings if `use_csv_target = True`

- **`runs`**: Initially empty. This is populated after each horizon finishes, typically with:
  - model parameter count (`n_params`)
  - training and evaluation wall time
  - metrics (for example RMSE/R²/NSE depending on your implementation)
  - the final hyperparameter dictionary used for that horizon

### Printed diagnostics

The final `print(...)` statements confirm the notebook is using the hardware you expect (example: GPU over CPU) before training begins.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# hardware_info = get_hardware_info(device) # commented for privacy

compute_summary = {
    "run_timestamp": datetime.utcnow().isoformat() + "Z",
    "run_name": run_name,
    "target_site": target_site,
    "target_variable": target_variable,
    "target_parameter_code": target_parameter_code,
    "target_units": target_units,
    "predictor_type": predictor_type,
    "start_date": start_date,
    "end_date": end_date,
    "tz": tz,
    "horizons": horizons,
    "site_selection_mode": site_selection_mode,
    "shap_mode": shap_mode,
    #"hardware_info:", hardware_info,
    print("device:", device),    
    "paths": {
        "examples_dir": str(resolve_under_project(project_root, examples_dir)),
        "project_root": str(project_root),
        "results_root": str(results_root),
        "predictor_json_root": str(predictor_json_root),
        "predictor_json_path": str(json_path),
        "full_shap_root": str(full_shap_root),
        "use_csv_target": bool(use_csv_target),
        "target_csv_path": str(csv_path) if use_csv_target else None,
        "target_csv_date_col": csv_date_col if use_csv_target else None,
        "target_csv_value_col": csv_value_col if use_csv_target else None,
    },
    "runs": {},
}

print("device:", device)
# print("hardware_info:", hardware_info)

## Run training and evaluation by horizon

Now we will run the full training and evaluation workflow separately for each forecast horizon listed in `horizons`. Each horizon is treated as its own unit with its own hyperparameters, predictor-site subset, output directory, and saved metrics.

### Step 1. Iterate over horizons

- The loop runs once per horizon in `horizons`.
- The printed line `Horizon H = XX` is a simple progress marker so you can track the run in notebook output.

### Step 2. Build the hyperparameter dictionary for this horizon

- **`hp`** starts as a copy of **`hp_defaults`** so each horizon begins from the same baseline.
- If the current horizon exists in **`hp_by_h`**, those values overwrite the defaults for that horizon.

### Step 3. Determine how many predictor sites to use (SHAP mode)

- **`n_sites`** is set from `n_shap_by_h[fh]` when available, otherwise it falls back to `default_n_shap`.
- This value is only relevant when `site_selection_mode = "from_shap"` (SHAP-ranked subset selection).

### Step 4. Select the allowed predictor sites

- **`allowed_sites`** is determined by `get_allowed_sites_for_horizon(...)`.
- This function enforces the site selection policy defined by **`site_selection_mode`**:
  - `"all"`: allow all predictor sites in the cached dataset.
  - `"from_shap"`: allow only the SHAP-ranked subset for the current horizon (read from `full_shap_root`).

The returned `allowed_sites` list is passed directly into training so the model only sees the intended predictors.

### Step 5. Create the output directory for this horizon

- **`out_dir = horizon_dir(results_root, fh)`** creates or points to a horizon-specific folder under the run directory.
- This keeps models, metrics, and artifacts separated cleanly by horizon.

### Step 6. Train and evaluate the model

- **`run_horizon(...)`** executes the end-to-end workflow for this horizon, including:
  - loading predictors from `data_files`
  - loading the target (from CSV if `use_csv_target = True`)
  - aligning to the `start_date` to `end_date` window in timezone `tz`
  - building sequences using `hp["sequence_length"]`
  - training the CNN–LSTM on `device`
  - evaluating performance and returning metrics

The returned dictionary **`out`** is expected to include:
- **`out["n_params"]`**: number of trainable model parameters
- **`out["train_time"]`**: training wall time (or recorded runtime)
- **`out["eval_time"]`**: evaluation runtime
- **`out["metrics"]`**: evaluation metrics for this horizon

### Step 7. Update the run summary

- **`update_compute_summary(...)`** writes per-horizon results into `compute_summary["runs"]`.
- The stored information typically includes:
  - horizon identifier (`fh`)
  - model size (`n_params`)
  - runtimes (`train_time`, `eval_time`)
  - evaluation metrics (`metrics`)
  - the exact hyperparameters used (`hp`)


In [ ]:
for fh in horizons:
    print(f"Horizon H = {fh:02d}")

    hp = dict(hp_defaults)
    if fh in hp_by_h:
        hp.update(hp_by_h[fh])

    n_sites = int(n_shap_by_h.get(fh, default_n_shap))

    allowed_sites = get_allowed_sites_for_horizon(
        fh,
        site_selection_mode=site_selection_mode,
        full_shap_root=full_shap_root,
        n_shap_by_h=n_shap_by_h,
        default_n_shap=default_n_shap,
    )

    out_dir = horizon_dir(results_root, fh)

    out = run_horizon(
        fh,
        data_files=data_files,
        use_csv_target=use_csv_target,
        target_site=target_site,
        target_parameter_code=target_parameter_code,
        start_date=start_date,
        end_date=end_date,
        tz=tz,
        hp=hp,
        device=device,
        allowed_sites=allowed_sites,
        csv_path=csv_path,
        csv_date_col=csv_date_col,
        csv_value_col=csv_value_col,
        shap_mode=shap_mode,
        site_selection_mode=site_selection_mode,
        full_shap_root=full_shap_root,
        n_sites=n_sites,
        out_dir=out_dir,
    )

    compute_summary = update_compute_summary(
        compute_summary,
        fh=fh,
        n_params=out["n_params"],
        train_time=out["train_time"],
        eval_time=out["eval_time"],
        metrics=out["metrics"],
        hp=hp,
    )


## Save outputs

Now we will write a JSON summary of the run to the run results directory.

### What gets saved

The `compute_summary` dictionary includes:

- **Run metadata**: timestamp, run name, target site/variable, predictor type, date range, timezone, horizons.
- **Path record**: resolved input and output paths used during the run.
- **Hardware info**: device selection (CPU or CUDA) and the associated hardware details.
- **Per-horizon results**: for each horizon, the recorded model size, runtimes, metrics, and the final hyperparameter dictionary used.

### Where it is saved

- The summary is written under **`results_root`**, which is the run-specific output directory created earlier.
- The final `print(...)` statement confirms the output directory location so you can quickly navigate to it and inspect files.


In [ ]:
save_compute_summary(results_root, compute_summary)
print("Saved compute summary to:", results_root)

If you have any questions or need help with implementation to your model, please do not hesitate to contact Caitlin R. R. Turner at cturn65@lsu.edu

## Acknowledgments

Research reported in this publication was supported by the US Department of Defense and Army Engineer Research and Development Center (ERDC) under Contract No. W912HZ2220005, the Gulf Research Program of the National Academies of Sciences, Engineering, and Medicine under award number SCON-10000883, and the NSF through Open Earthscape (Collaborative Research: Frameworks: OpenEarthscape, Transformative Cyberinfrastructure for Modeling and Simulation in the Earth-Surface Science Communities) award No. 2104102.